# 0.7 - XGBoost Floor Evaluation

Evaluates the 0.5 model on the test observations by floor of the measured room, at the thresholds matched to the radon map legend boundaries in 0.5.1 (confusion matrix at the decision threshold).

## Setup

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score

from evaluation import load_test_set, operating_points, plot_confusion_matrix, plot_precision_recall, regression_scores

ROOT = '../../'
DIR = '../../models/'

## Data and model

In [ ]:
X_test, y_test = load_test_set(ROOT)
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)

model = xgb.XGBRegressor()
model.load_model(f'{DIR}0.5-xgboost-log-model.json')
y_pred = pd.Series(np.expm1(model.predict(preprocessor.transform(X_test))), index=y_test.index)

# Settings chosen in 0.5.1
with open(f'{DIR}0.5-decision-threshold.json') as file:
    settings = json.load(file)
RADON_REFERENCE_LEVEL, DECISION_THRESHOLD = settings['reference_level'], settings['decision_threshold']
# Thresholds matched to the recall of the radon map at its legend boundaries; the decision threshold is the > 1 % one
THRESHOLDS = {f'Map {cutoff}': threshold for cutoff, threshold in settings['map_matched_thresholds'].items()}

## Evaluation

In [ ]:
def subset_summary(name, mask):
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    return {
        'Subset': name,
        'Observations': int(mask.sum()),
        'Prevalence (%)': round(exceeds.mean() * 100, 1),
        **regression_scores(y_test[mask], y_pred[mask]),
        'Average precision': round(average_precision_score(exceeds, y_pred[mask]), 3),
    }


def subset_operating_points(name, mask):
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    points = operating_points(exceeds, y_pred[mask], THRESHOLDS)
    points.insert(0, 'Subset', name)
    return points


def plot_subset(name, mask):
    exceeds = y_test[mask] >= RADON_REFERENCE_LEVEL
    plot_precision_recall(exceeds, y_pred[mask], f'Precision–recall - {name}')
    plot_confusion_matrix(exceeds, y_pred[mask] >= DECISION_THRESHOLD, f'{name} - threshold {DECISION_THRESHOLD:.0f} Bq/m3', RADON_REFERENCE_LEVEL)

In [ ]:
floor = X_test['Rdn - Etage']
FLOOR_GROUPS = {'Floor < 0': floor < 0, 'Floor = 0': floor == 0, 'Floor ≥ 1': floor >= 1}

pd.DataFrame([subset_summary(name, mask) for name, mask in FLOOR_GROUPS.items()])

In [ ]:
pd.concat([subset_operating_points(name, mask) for name, mask in FLOOR_GROUPS.items()], ignore_index=True)

In [ ]:
for name, mask in FLOOR_GROUPS.items():
    plot_subset(name, mask)